In [1]:
from langchain_community.document_loaders import PyPDFLoader
import os

data_dir = "../data"
pdf_files = [f for f in os.listdir(data_dir) if f.endswith(".pdf")]
print(pdf_files)

all_docs = []
for file in pdf_files:
    loader = PyPDFLoader(os.path.join(data_dir, file))
    docs = loader.load()
    all_docs.extend(docs)

print(f"Total pages loaded: {len(all_docs)}")
print(all_docs[0].page_content[:500])

C:\Users\admin\AppData\Local\Temp\ipykernel_25820\2894779341.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


['AI_Based_Behaviour_Trees_and_EQS_for_Realistic_NPC_Decision_Making_in_Historical_Games.pdf', 'Payslip.pdf', 'The_Illicit_Happiness_of_Other_People.pdf']
Total pages loaded: 12
An Adaptive Behaviour Tree and EQS-Based
Framework for Realistic NPC Decision-Making in
Historical Game Environments
Sara Jamil 1, Devansh Paurya 2, Gobinda Debnath 3, Aditya Kumar Khatua 4, Yogiraj Anil Bhale 5
1234 Department of Computer Science and Engineering,
Chandigarh University, Mohali, India
5Assistant Professor, Department of Computer Science and Engineering,
Chandigarh University, Mohali, India
1sarajamil385@gmail.com
2devansh582004@gmail.com
3gobify2402@gmail.com
4adityakhatua231204@


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(all_docs)
print(f"Total chunks: {len(chunks)}")
print(chunks[0].page_content)
print(chunks[0].metadata)

Total chunks: 97
An Adaptive Behaviour Tree and EQS-Based
Framework for Realistic NPC Decision-Making in
Historical Game Environments
Sara Jamil 1, Devansh Paurya 2, Gobinda Debnath 3, Aditya Kumar Khatua 4, Yogiraj Anil Bhale 5
1234 Department of Computer Science and Engineering,
Chandigarh University, Mohali, India
5Assistant Professor, Department of Computer Science and Engineering,
Chandigarh University, Mohali, India
1sarajamil385@gmail.com
2devansh582004@gmail.com
3gobify2402@gmail.com
{'producer': 'pdfTeX-1.40.27', 'creator': 'TeX', 'creationdate': '2026-06-24T18:06:12+00:00', 'moddate': '2026-06-24T18:06:12+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'trapped': '/False', 'source': '../data\\AI_Based_Behaviour_Trees_and_EQS_for_Realistic_NPC_Decision_Making_in_Historical_Games.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}


In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Quick test
test_vector = embedding_model.embed_query("What is machine learning?")
print(len(test_vector))
print(test_vector[:5])

C:\Users\admin\AppData\Local\Temp\ipykernel_25820\523583014.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

a:\Projects\document-qa\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\admin\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (7 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

384
[-0.019954534247517586, 0.009878017008304596, 0.010249611921608448, 0.029553672298789024, 0.02718639001250267]


In [6]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="../chroma_db"
)

print("Vector store created with", vectorstore._collection.count(), "chunks")

Vector store created with 97 chunks


In [7]:
query = "What is the salary mentioned in the payslip?"
results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(doc.page_content[:200])
    print(doc.metadata.get('source'))
    print()

--- Result 1 ---
Advance Statutory Bonus 906.00 906.00 0.00 906.00
Deductions Amount
LWF Deduction 5.00
Provident Fund 1,272.00
ESIC 80.00
Gross Earnings 17,873.00 Total Deductions 1,357.00
In words ( ) : Sixteen Thou
../data\Payslip.pdf

--- Result 2 ---
Advance Statutory
Bonus
664.00 7,248.00 906.00 8,818.00
Salary For The Year 173,964.00
Gross Salary 173,964.00
Gross Taxable Income 173,964.00
Less : Standard Deduction 75,000.00
Net Taxable Income (R
../data\Payslip.pdf

--- Result 3 ---
Less Tax Deducted by Previous Employer 0.00
Balance Tax Payable/Refundable 0.00
Average Tax Payable per Month 0.00
Details Of Exemption U/S 10
Gratuity Exemption 0.00
Monthly Tax deducted in Salary
Mo
../data\Payslip.pdf



In [8]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.1:8b")

# Quick test
response = llm.invoke("Say hello in one sentence.")
print(response)

Hello, how are you today?


In [12]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """Answer the question based only on the following context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("What is the main contribution of the AI behaviour trees paper?")
print(answer)

The main contribution of the AI behaviour trees paper is that Behaviour Trees (BTs) have become an efficient alternative for decision-making in AI systems, particularly in game development, due to their modularity, hierarchical control flow, and maintainability.


In [14]:
def rag_query(question):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    answer = rag_chain.invoke(question)
    print("Answer:", answer)
    print("\nSources:")
    for doc in docs:
        print("-", doc.metadata.get("source"), "| Page:", doc.metadata.get("page"))
    print("\n" + "="*50 + "\n")
    return answer

rag_query("What is the main contribution of the AI behaviour trees paper?")
rag_query("What is mentioned in the payslip?")
rag_query("What is the book 'The Illicit Happiness of Other People' about?")

Answer: The main contribution of the AI behaviour trees paper is that they have become an efficient alternative for AI systems, particularly for Non-Player Characters (NPCs), due to their modularity, hierarchical control flow, and maintainability.

Sources:
- ../data\AI_Based_Behaviour_Trees_and_EQS_for_Realistic_NPC_Decision_Making_in_Historical_Games.pdf | Page: 0
- ../data\AI_Based_Behaviour_Trees_and_EQS_for_Realistic_NPC_Decision_Making_in_Historical_Games.pdf | Page: 7
- ../data\AI_Based_Behaviour_Trees_and_EQS_for_Realistic_NPC_Decision_Making_in_Historical_Games.pdf | Page: 6


Answer: According to the provided context, the following information is mentioned in the payslip:

1. Basic Salary: 7,775.00
2. House Rent Allowance: 4,668.00
3. Advance Statutory Bonus: 906.00
4. Gross Earnings: 17,873.00
5. Total Deductions: 1,357.00
6. Net Salary: 16,516.00

Sources:
- ../data\Payslip.pdf | Page: 0
- ../data\Payslip.pdf | Page: 0
- ../data\Payslip.pdf | Page: 0


Answer: The book "The

'The book "The Illicit Happiness of Other People" by Manu Joseph appears to be about a man who leaves his wife and child and lives in a new place, leading to a complex exploration of character, culture, and religion. The book is also described as a "cocktail" of these elements, suggesting a rich and layered narrative.'

In [15]:
test_cases = [
    {"question": "What is the main contribution of the AI behaviour trees paper?", "expected_source": "AI_Based_Behaviour_Trees"},
    {"question": "What is the net salary in the payslip?", "expected_source": "Payslip"},
    {"question": "What is the book about?", "expected_source": "The_Illicit_Happiness"},
]

correct = 0
for case in test_cases:
    docs = retriever.invoke(case["question"])
    top_source = docs[0].metadata.get("source", "")
    is_correct = case["expected_source"] in top_source
    correct += is_correct
    print(f"Q: {case['question']}")
    print(f"Expected: {case['expected_source']} | Got: {top_source} | {'✓' if is_correct else '✗'}\n")

print(f"Retrieval accuracy: {correct}/{len(test_cases)}")

Q: What is the main contribution of the AI behaviour trees paper?
Expected: AI_Based_Behaviour_Trees | Got: ../data\AI_Based_Behaviour_Trees_and_EQS_for_Realistic_NPC_Decision_Making_in_Historical_Games.pdf | ✓

Q: What is the net salary in the payslip?
Expected: Payslip | Got: ../data\Payslip.pdf | ✓

Q: What is the book about?
Expected: The_Illicit_Happiness | Got: ../data\The_Illicit_Happiness_of_Other_People.pdf | ✓

Retrieval accuracy: 3/3


In [16]:
import mlflow

mlflow.set_experiment("document-qa-rag")

with mlflow.start_run(run_name="rag-baseline"):
    mlflow.log_params({
        "llm_model": "llama3.1:8b",
        "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
        "chunk_size": 500,
        "chunk_overlap": 50,
        "retrieval_k": 3,
        "vector_store": "ChromaDB",
        "num_documents": len(pdf_files),
        "num_chunks": len(chunks)
    })
    mlflow.log_metric("retrieval_accuracy", correct / len(test_cases))

2026/09/30 02:56:06 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/30 02:56:06 INFO mlflow.store.db.utils: Updating database tables
2026/09/30 02:56:07 INFO mlflow.tracking.fluent: Experiment with name 'document-qa-rag' does not exist. Creating a new experiment.
